In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langgraph.graph import StateGraph,START, END
from typing import TypedDict
from IPython.display import Image
import os

d:\All Rounder Development\Artificial Intelligence\Agentic AI\LangGraph\Lang-Graph-From-Zero-To-Hero\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
d:\All Rounder Development\Artificial Intelligence\Agentic AI\LangGraph\Lang-Graph-From-Zero-To-Hero\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['HF_HOME'] = r'D:\huggingface_cache\chat_models\Qwen3-0.6B'

llm = HuggingFacePipeline.from_model_id(
    model_id='Qwen/Qwen3-0.6B',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1711.19it/s]


In [3]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Please create the outline of the following title - {title}'
    outline = model.invoke(prompt)
    state['outline'] = outline
    return state

In [5]:
def create_contnet(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    prompt = f'Please create the detailed content of the following title - {title} and outline - {outline}'
    outline = model.invoke(prompt)
    state['outline'] = outline
    return state

In [7]:
# create our graph
graph = StateGraph(BlogState)
# add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_contnet', create_contnet)
# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_contnet')
graph.add_edge('create_contnet', END)
# compile
workflow = graph.compile()

In [9]:
# execute

intial_state = {'title': 'The Rise of AI in Pakistan'}

final_state = workflow.invoke(intial_state)

print(final_state)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'title': 'The Rise of AI in Pakistan', 'outline': AIMessage(content='<|im_start|>user\nPlease create the detailed content of the following title - The Rise of AI in Pakistan and outline - content=\'<|im_start|>user\\nPlease create the outline of the following title - The Rise of AI in Pakistan<|im_end|>\\n<|im_start|>assistant\\n<think>\\nOkay, the user wants an outline for the title "The Rise of AI in Pakistan." Let me start by breaking down the title. "The Rise" suggests a growing trend, "AI" refers to artificial intelligence, and "Pakistan" is the country.\\n\\nFirst, I need to structure the outline properly. Typically, such titles have a main subject, a description, and possibly a context or significance. Let me think about the key points. \\n\\nThe title is about the growth of AI in Pakistan. So the outline should mention the country\\\'s situation, the development of AI, and the impact. Maybe start with the historical context, then the current state, followed by the significance

In [11]:
Image(workflow.get_graph().draw_mermaid_png())

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`